In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [2]:
train = pd.read_csv('./data/train_reviews.csv')
test = pd.read_csv('./data/test_reviews.csv')
negocios = pd.read_csv('./data/negocios.csv')
usuarios = pd.read_csv('./data/usuarios.csv')

C:\Users\Lluis\AppData\Local\Temp\ipykernel_24784\4243529203.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios = pd.read_csv('./data/usuarios.csv')


In [3]:
def generateSubmision(test_preds, name):
    submition = pd.DataFrame()
    submition['review_id'] = test['review_id']
    submition['stars'] = test_preds
    submition.to_csv(f'./data/submissions/submission_{name}.csv', index=False)

In [3]:
from transformers import pipeline

# Load sentiment-analysis pipeline (default model: distilbert-base-uncased-finetuned-sst-2-english)
classifier = pipeline("sentiment-analysis")

# Example
text = "I love how simple Hugging Face makes it to use transformers!"
result = classifier(text)
print(result)


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\Lluis\anaconda3\envs\Env\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lluis\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9997546076774597}]


In [7]:
test['text'][0]

"Amazing coffee and chill atmosphere. The staff was friendly and it's located right by my condo. We needed a cute coffee place like this close by. Glad to welcome them as our neighbors."

In [9]:
result = classifier(test['text'][1])
print(result)

[{'label': 'POSITIVE', 'score': 0.9861772656440735}]


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device =0)

print(classifier("Este producto es una maravilla. Lo recomiendo mucho."))


Device set to use cuda:0


[{'label': '5 stars', 'score': 0.8805814981460571}]


In [5]:
results = classifier(list(test['text'][0:5].values))
print(results)

[{'label': '5 stars', 'score': 0.8224518299102783}, {'label': '4 stars', 'score': 0.35095998644828796}, {'label': '5 stars', 'score': 0.8513038754463196}, {'label': '4 stars', 'score': 0.4647212624549866}, {'label': '1 star', 'score': 0.9642598032951355}]


In [6]:
test.shape

(414765, 8)

In [8]:
results = classifier(test["text"].head(1000).tolist(), batch_size=64, truncation=True)

# Convert results (list of dicts) to DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df['stars'] = results_df['label'].apply(lambda x: int(x.split(' ')[0]))
results_df.head()

,label,score,stars
0,1 star,0.705297,1
1,2 stars,0.445256,2
2,5 stars,0.731513,5
3,2 stars,0.444388,2
4,4 stars,0.700352,4


In [ ]:
generateSubmision(results_df['stars'], "transformer-bert-base-sentiment")

In [ ]:
import torch
from torch.nn.functional import softmax
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Inference settings
batch_size =64
texts = test["text"].tolist()
all_labels = []
all_scores = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]

    # Tokenize batch
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)

    # No gradient computation for inference
    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits, dim=1)
        scores, labels = torch.max(probs, dim=1)

    # Convert to CPU and numpy
    all_labels.extend([model.config.id2label[label.item()] for label in labels])
    all_scores.extend([score.item() for score in scores])

# Add predictions to DataFrame
test["label"] = all_labels
test["score"] = all_scores


  0%|          | 0/6481 [00:00<?, ?it/s]

100%|██████████| 6481/6481 [1:54:49<00:00,  1.06s/it]


AttributeError: 'float' object has no attribute 'split'

In [ ]:
test["label"] = test["label"].apply(lambda x: int(x.split(' ')[0]))
print(test[['label','score']].head())

In [21]:
test

,review_id,user_id,business_id,useful,funny,cool,text,date,label,score
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,5,0.822452
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,4,0.350960
2,seR2KhblYMWg-k9zzN6aYA,hF68a0mpu97u0oaryFYhyg,_RG4IByyBR528CMc7DefJA,2,0,0,Came here when my kitten got very sick by the ...,2015-09-06 15:29:02,5,0.851304
3,BToo00Fi5pfJFA5MI2HM5g,G4yX5Q1tFfwSucFOmiyjdA,xxlbRiWWQkk-6LST3Hd12g,2,0,0,So I'll preface by saying we did have an overa...,2015-09-14 00:49:17,4,0.464722
4,FHJAzi1imodBit3RWK7zQA,Srqi1xb7exdB9uRHxDeEkw,LgGqdFLD7-ca0Z9F_q4Fuw,0,0,0,This place is a joke. Worst bar service ever. ...,2015-07-24 01:03:40,1,0.964260
...,...,...,...,...,...,...,...,...,...,...
414760,5VF1OSReEKAJS3z7lc0VRg,2whTgW_TDIPh7qRdTxzm0Q,W4ZEKkva9HpAdZG88juwyQ,0,0,0,I ate at Mr. B's for a milestone birthday dinn...,2011-02-23 01:53:33,5,0.488763
414761,N_yTSNrcGzGUSFPyIhAZ-g,1k3HsUHlVB1q5RfWKAkusw,kqtyPqKBC7TyoIn9Ml3nsA,0,0,0,The employees are so nice and the service was ...,2019-07-22 17:20:54,5,0.931770
414762,zQLW0mkqDtmseg8pl-zbNA,vXYcOtA-rogHpBFZEPKbYg,_s_51Vmq1LVQnGFSa-KfUw,0,0,0,"I came here awhile back, opening night to the ...",2012-07-25 16:47:37,3,0.808624
414763,DspB5yysBWRlFe01j_LzeA,3z2FQKEmFZ7zKuFIG1B1qw,-BjXcRcvDqOQ23eUKx6hjg,4,0,1,"Great, great spot in what looks to be an up an...",2011-08-22 13:31:40,5,0.802768


In [22]:
generateSubmision(test['label'], "transformer-bert-base-sentiment")